In [3]:
import joblib
import re
import numpy as np
import xgboost as xgb

# Load the trained models
rf_model = joblib.load('./tello_command_model_rf.pkl')  # Replace with actual RF model path
xgb_model = joblib.load('./tello_command_model_xgb.pkl')  # Replace with actual XGBoost model path
svm_model = joblib.load('./tello_command_model_svm.pkl')  # Replace with actual SVM model path

# Define a feature extraction function (same as during training)
def extract_features(text, x_value):
    features = [
        len(text),                      # Length of the sentence
        len(text.split()),               # Number of tokens
        1 if bool(x_value) else 0,       # Whether there is an 'x' value (slot)
    ]
    
    keywords = ["takeoff", "land", "streamon", "streamoff", "up x", "down x", "left x", "right x", 
                "forward x", "back x", "battery?"]
    for keyword in keywords:
        features.append(int(keyword in text.lower()))  # Presence of keyword
    
    # Ensure the number of features is exactly 14
    if len(features) != 14:
        raise ValueError(f"Expected 14 features, but got {len(features)}.")
    
    return features

# Define the prediction function for all three models
def predict_command(human_command):
    # Extract the x value (if any)
    x_value = re.findall(r"\d+", human_command)
    x_value = x_value[0] if x_value else None
    
    # Extract features for the input command
    features = extract_features(human_command, x_value)
    
    # Convert features into a DMatrix for XGBoost
    xgb_dmatrix = xgb.DMatrix([features])

    # Predict with RandomForest
    rf_pred = rf_model.predict([features])[0]
    
    # Predict with XGBoost
    xgb_pred = xgb_model.predict(xgb_dmatrix)[0]
    
    # Predict with SVM
    svm_pred = svm_model.predict([features])[0]
    
    return rf_pred, xgb_pred, svm_pred

# Example: Test the function with a random human command
random_human_command = "Automatically land"
rf_pred, xgb_pred, svm_pred = predict_command(random_human_command)

print(f"Human Command: {random_human_command}")
print(f"Predicted SDK Command (Random Forest): {rf_pred}")
print(f"Predicted SDK Command (XGBoost): {xgb_pred}")
print(f"Predicted SDK Command (SVM): {svm_pred}")


Human Command: Automatically land
Predicted SDK Command (Random Forest): takeoff
Predicted SDK Command (XGBoost): 10.0
Predicted SDK Command (SVM): 1
